## Generative Unsupervised Defect Detection Pipeline.

#### Phase 1: Data Acquisition & The Definition of "Normal"

We must first secure our industrial benchmark dataset (MVTec AD) and establish a baseline.

Data Setup: Programmatically download and extract the MVTec AD dataset directly within Python using the anomalib library.

EDA (Exploratory Data Analysis): Filter the dataset strictly for "good" (defect-free) samples of a complex industrial component (e.g., metal nuts or transistors).

Deliverable: A pristine visual grid of perfect manufacturing components, establishing the baseline distribution for our unsupervised model.

#### Phase 2: The Synthetic Defect Engine (Stable Diffusion + ControlNet)


Since we lack anomalous data, we will act as the "rogue manufacturer" and generate our own defects.

Edge Preservation: Run a pristine component through a Canny Edge Detector to extract its physical geometry. This ensures our AI-generated defects don't alter the actual shape of the part.

Inpainting & Synthesis: Pass the edge map into ControlNet, paired with an Inpainting Stable Diffusion pipeline. We will use text prompts (e.g., "deep rust," "jagged scratch") to synthesize highly realistic anomalies exactly where we want them.

Deliverable: A side-by-side visualization showing the pristine industrial part and our synthetically damaged variant, complete with its corresponding generated ground-truth mask.

#### Phase 3: Unsupervised Feature Extraction (PatchCore)

We will train an anomaly detector strictly on the clean data from Phase 1.

Memory Bank Creation: Pass the clean images through a pre-trained feature extractor (like Wide ResNet-50) to build a multi-scale representation of normal patches.

Coreset Subsampling: Because industrial components have high structural redundancy, we compress these features into a lightweight "memory bank" (coreset) for lightning-fast inference.

Deliverable: An interactive or plotted representation showing the spatial feature extraction process, proving the model only knows what a "perfect" part looks like.

#### Phase 4: Inference & Heatmap Localization


The grand finale. We will test if our unsupervised PatchCore model can catch the synthetic defects we created in Phase 2.

Distance Metric: Pass our synthetically damaged part into the network and calculate the nearest-neighbor distance between its local patches and our pristine memory bank.

Thresholding: Establish a mathematical anomaly score to classify the part as PASS or FAIL.

Deliverable: A stunning, highly localized color heatmap superimposed over the damaged part, glowing red exactly where our synthetic rust or scratch was generated.

## Phase 1: Data Acquisition & The Definition of "Normal".

Before we can detect anomalies or generate synthetic defects, we must establish a mathematical and visual baseline of a "perfect" industrial component.

We will use the MVTec AD dataset, which is the gold standard benchmark for unsupervised anomaly detection in industrial inspection. The dataset contains over 5,000 high-resolution color images divided into 15 object and texture categories. Crucially for our unsupervised approach, it comprises a set of defect-free training images (the "normal" baseline) and a separate test set containing various defects. It is released under the CC BY-NC-SA 4.0 license.

Instead of downloading the entire 5GB dataset, we will write a script to programmatically download just the specific component category we need (e.g., a glass bottle) directly into our Kaggle environment.

In [ ]:
#!pip install diffusers transformers accelerate

import pandas as pd
import numpy as np
import cv2
import torch
import PIL
import os
import json
import matplotlib.pyplot as plt
import warnings

from PIL import Image, ImageDraw
from pathlib import Path
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, UniPCMultistepScheduler

warnings.filterwarnings('ignore')


def execute_phase1(dataset_path='/kaggle/input/datasets/ipythonx/mvtec-ad/bottle'):
    print("Initializing Phase 1: Data Exploratory Analysis...")
    
    root_dir = Path(dataset_path)
    
    # Target strictly the 'good' (defect-free) training images
    train_good_dir = root_dir / "train" / "good"
    
    if not train_good_dir.exists():
        print(f"Error: Could not find path {train_good_dir}. Please verify the Kaggle input directory name.")
        return None

    # Grab all PNG images in the 'good' training folder
    image_paths = sorted(list(train_good_dir.glob("*.png")))
    
    print(f"Found {len(image_paths)} pristine training images.")
    
    # Visualize the definition of "Normal"
    num_to_show = min(4, len(image_paths))
    fig, axes = plt.subplots(1, num_to_show, figsize=(16, 5))
    
    for i in range(num_to_show):
        img = Image.open(image_paths[i])
        axes[i].imshow(img)
        axes[i].set_title(f"Pristine Baseline #{i+1}")
        axes[i].axis("off")
        
    plt.suptitle("Phase 1: Defining 'Normal' (Defect-Free Components)", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Return the first pristine image path to use in Phase 2
    return image_paths[0]

if __name__ == "__main__":
    pristine_image_path = execute_phase1('/kaggle/input/datasets/ipythonx/mvtec-ad/bottle')

## Phase 2: The Synthetic Defect Engine.

If we ask a standard Generative AI to "add a crack to this bottle," it will likely hallucinate an entirely new bottle with a different shape and background. In manufacturing, precision is everything. We cannot alter the geometry of the part or the background scene.

To solve this, we combine two powerful models:

Inpainting: Restricts the AI to only modify pixels inside a specific "mask" (the area we want damaged).

ControlNet: Extracts the physical geometry (edges) of the original bottle and forces the AI to respect those exact structural boundaries during generation.

#### 1. Environment Setup
We will use the Hugging Face diffusers library.

In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, UniPCMultistepScheduler

def execute_phase2(image_path):
    print("initializing phase 2: Synthetic Defect Generation...")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load the pristine image
    init_image = Image.open(image_path).convert("RGB").resize((512, 512))

    # Extract geometry (Canny edges)
    image_np = np.array(init_image)
    low_threshold, high_threshold = 100, 200
    edges = cv2.Canny(image_np, low_threshold, high_threshold)
    control_image = Image.fromarray(np.stack([edges]*3, axis=2))

    # Create a defect mask (single-channel L)
    mask_image = Image.new("L", (512, 512), 0)
    draw = ImageDraw.Draw(mask_image)
    draw.line((256, 150, 256, 350), fill=255, width=40)

    # Load models
    print("Loading ControlNet and inpainting weights")
    torch_dtype = torch.float16 if device == "cuda" else torch.float32

    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-canny", torch_dtype=torch_dtype
    )
    pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        controlnet=controlnet,
        torch_dtype=torch_dtype
    )

    # Move pipeline to device and optimize memory usage
    pipe = pipe.to(device)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.enable_attention_slicing()   # reduces VRAM usage

    # If accelerate is installed and an accelerator is present, you can enable CPU offload.
    # But avoid calling enable_model_cpu_offload() when accelerator is not available.
    try:
        # this will raise if accelerate/accelerator is not configured
        pipe.enable_model_cpu_offload()
    except Exception:
        pass

    # Generate the synthetic anomaly
    prompt = "a deep, ugly, shattered glass crack, highly detailed, realistic defect"
    negative_prompt = "cartoon, illustration, changing shape, out of frame"

    print(f"Generating Synthetic anomaly with prompt: {prompt}")

    # Use autocast on CUDA for faster half precision inference
    if device == "cuda":
        with torch.autocast(device_type="cuda"):
            out = pipe(
                prompt=prompt,
                image=init_image,
                mask_image=mask_image,
                control_image=control_image,
                negative_prompt=negative_prompt,
                controlnet_conditioning_scale=0.8,
                num_inference_steps=25
            )
    else:
        out = pipe(
            prompt=prompt,
            image=init_image,
            mask_image=mask_image,
            control_image=control_image,
            negative_prompt=negative_prompt,
            controlnet_conditioning_scale=0.8,
            num_inference_steps=25
        )

    generated_image = out.images[0]

    # Visualization
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(init_image); axes[0].set_title("Original Pristine Part")
    axes[1].imshow(control_image); axes[1].set_title("Geometric Constraints Canny")
    axes[2].imshow(mask_image, cmap="gray"); axes[2].set_title("Target Anomaly Zone Mask")
    axes[3].imshow(generated_image); axes[3].set_title("Synthesized Defect")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


execute_phase2(pristine_image_path)


## Phase 3: Unsupervised Feature Extraction (PatchCore).

In standard AI projects, you pass an image into a Convolutional Neural Network (CNN) and it outputs a prediction (like "Cat" or "Dog"). We are going to use a CNN entirely differently.

We will push our perfect, defect-free bottles through a pre-trained Wide ResNet-50, but we will chop off the classification head. Instead, we will intercept the data halfway through the network (at layers 2 and 3). At this stage, the network has calculated rich, multi-scale mathematical representations of the image's textures and edges (known as feature maps).

We will chop these feature maps into tiny patches and throw them all into a giant "Memory Bank." Because a memory bank of millions of patches will crash your RAM, PatchCore uses Coreset Subsampling—a greedy algorithmic approach that compresses the memory bank while preserving the complete spatial distribution of what "normal" looks like.

In [ ]:
import torch
import torchvision.transforms as T
from torchvision.models import wide_resnet50_2, Wide_ResNet50_2_Weights
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from sklearn.decomposition import PCA

class PatchCoreExtractor:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Load Wide ResNet-50
        print("Loading Wide ResNet-50 Feature Extractor...")
        self.model = wide_resnet50_2(weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(self.device)
        self.model.eval()
        
        # Dictionaries to hold our intercepted features
        self.features = {}
        
        # Register hooks to intercept data at Layer 2 and Layer 3
        def get_features(name):
            def hook(model, input, output):
                self.features[name] = output.detach()
            return hook
            
        self.model.layer2.register_forward_hook(get_features('layer2'))
        self.model.layer3.register_forward_hook(get_features('layer3'))
        
        self.transform = T.Compose([
            T.Resize((256, 256)),
            T.CenterCrop(224),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        self.memory_bank = []

    def extract_patch_features(self, image_tensor):
        """Passes image through CNN and extracts multi-scale patch features."""
        with torch.no_grad():
            _ = self.model(image_tensor)
            
        # Get intermediate features
        feat2 = self.features['layer2']
        feat3 = self.features['layer3']
        
        # Average pooling to smooth features 
        feat2 = F.avg_pool2d(feat2, 3, 1, 1)
        feat3 = F.avg_pool2d(feat3, 3, 1, 1)
        
        # Resize Layer 3 features to match Layer 2 spatial dimensions
        feat3 = F.interpolate(feat3, size=feat2.shape[-2:], mode="bilinear", align_corners=False)
        
        # Concatenate them along the channel dimension
        patch_features = torch.cat([feat2, feat3], dim=1)
        
        # Reshape to (Batch * Height * Width, Channels)
        patch_features = patch_features.permute(0, 2, 3, 1).reshape(-1, patch_features.shape[1])
        return patch_features.cpu().numpy()

def execute_phase3(dataset_path='/kaggle/input/datasets/ipythonx/mvtec-ad/bottle'):
    print("Initializing Phase 3: PatchCore Memory Bank Generation...\n")
    
    extractor = PatchCoreExtractor()
    
    # Load 20 pristine training images to build our baseline
    train_good_dir = Path(dataset_path) / "train" / "good"
    image_paths = sorted(list(train_good_dir.glob("*.png")))[:20]
    
    all_features = []
    
    print(f"Extracting features from {len(image_paths)} pristine images...")
    for path in image_paths:
        img = Image.open(path).convert("RGB")
        img_tensor = extractor.transform(img).unsqueeze(0).to(extractor.device)
        
        patch_feats = extractor.extract_patch_features(img_tensor)
        all_features.append(patch_feats)
        
    # Combine all patch features from all images
    all_features = np.vstack(all_features)
    print(f"Total raw patches extracted: {all_features.shape[0]} patches, {all_features.shape[1]} dimensions.")
    
    # Coreset Subsampling (Simulated via random selection for notebook memory efficiency)
    # In production, this uses K-Center Greedy algorithms to preserve feature distribution
    coreset_ratio = 0.05  # Keep 5% of the most representative patches
    num_coreset = int(all_features.shape[0] * coreset_ratio)
    
    np.random.seed(42)
    indices = np.random.choice(all_features.shape[0], num_coreset, replace=False)
    memory_bank = all_features[indices]
    
    print(f"Memory Bank optimized via Coreset Subsampling: {memory_bank.shape[0]} patches saved.\n")
    
    # Visualization: Plot the "Concept of Normal" in 2D space using PCA
    print("Visualizing the Memory Bank feature space...")
    pca = PCA(n_components=2)
    reduced_features = pca.fit_transform(memory_bank)
    
    plt.figure(figsize=(10, 6))
    plt.scatter(reduced_features[:, 0], reduced_features[:, 1], alpha=0.5, c='blue', s=10)
    plt.title("Phase 3: The Mathematical Definition of 'Normal'")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.grid(True)
    plt.show()
    
    return extractor, memory_bank

if __name__ == "__main__":
    extractor, memory_bank = execute_phase3()

## Phase 4: Inference & Heatmap Localization. 

This is the grand finale.

We are going to test our unsupervised model by passing an anomalous image into our PatchCoreExtractor. The model will extract patches from the damaged image and search for the closest matching patches in our perfect memory_bank.

Because our memory bank only contains features from perfect components, a deep scratch or a broken edge will produce feature patches that have no mathematically similar neighbors. The anomaly score is taken as the maximum distance between the test patch and its nearest neighbor in the memory bank. We can then reshape these mathematical distances into a spatial heatmap.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

def execute_phase4(extractor, memory_bank, dataset_path='/kaggle/input/datasets/ipythonx/mvtec-ad/bottle'):
    print("Initializing Phase 4: Inference & Heatmap Localization...")
    
    # 1. Fit Nearest Neighbors on the Memory Bank
    print("Fitting Nearest Neighbors algorithm on the Coreset Memory Bank...")
    knn = NearestNeighbors(n_neighbors=9, algorithm='ball_tree') 
    knn.fit(memory_bank)
    
    # 2. Load an Anomalous Test Image
    # We will use a real defect from the test set for reliable validation
    test_broken_dir = Path(dataset_path) / "test" / "broken_large"
    test_image_path = list(test_broken_dir.glob("*.png"))[0]
    
    img = Image.open(test_image_path).convert("RGB")
    img_tensor = extractor.transform(img).unsqueeze(0).to(extractor.device)
    
    # 3. Extract Features for the Test Image
    print(f"Extracting patches from test image: {test_image_path.name}...")
    test_features = extractor.extract_patch_features(img_tensor)
    
    # 4. Calculate Nearest Neighbor Distances (Anomaly Scoring)
    distances, _ = knn.kneighbors(test_features)
    
    # Average the distance of the k nearest neighbors for stability
    patch_scores = distances.mean(axis=1) 
    
    # The global image anomaly score is the maximum patch score
    image_score = patch_scores.max()
    
    # 5. Reshape into a Spatial Heatmap
    # For a 224x224 input image, Layer 2 of Wide ResNet-50 produces a 28x28 feature grid
    feature_map_size = 28 
    anomaly_map = patch_scores.reshape(feature_map_size, feature_map_size)
    
    # 6. Upsample and Smooth the Heatmap
    # Resize back to the 224x224 image space
    anomaly_map_resized = cv2.resize(anomaly_map, (224, 224), interpolation=cv2.INTER_CUBIC)
    anomaly_map_smoothed = cv2.GaussianBlur(anomaly_map_resized, (15, 15), 0)
    
    # Normalize to [0, 1] for visualization
    min_val, max_val = anomaly_map_smoothed.min(), anomaly_map_smoothed.max()
    anomaly_map_normalized = (anomaly_map_smoothed - min_val) / (max_val - min_val)
    
    # Convert to a Jet colormap (Glowing Red = Anomaly)
    heatmap = np.uint8(255 * anomaly_map_normalized)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    # Crop/Normalize the original image to perfectly match the 224x224 extractor transform
    original_cropped = extractor.transform(img).permute(1, 2, 0).cpu().numpy()
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    original_cropped = np.clip(std * original_cropped + mean, 0, 1)
    original_cropped_uint8 = np.uint8(255 * original_cropped)
    
    # Create the overlay
    overlay = cv2.addWeighted(original_cropped_uint8, 0.5, heatmap, 0.5, 0)
    
    # 7. Visualization
    print(f"Global Anomaly Score: {image_score:.2f}")
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    axes[0].imshow(original_cropped)
    axes[0].set_title("1. Anomalous Test Image")
    
    axes[1].imshow(anomaly_map_normalized, cmap='jet')
    axes[1].set_title("2. Raw Anomaly Heatmap")
    
    axes[2].imshow(overlay)
    axes[2].set_title(f"3. Localized Defect Overlay")
    
    for ax in axes:
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Ensure extractor and memory_bank from Phase 3 are passed in!
    execute_phase4(extractor, memory_bank)